### 흥국생명 실손보험 챗봇

In [7]:
# Agent + RAG
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain_tavily import TavilySearch
from datetime import datetime

# RAG 관련
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool

from dotenv import load_dotenv
load_dotenv()

today = datetime.today().strftime('%Y-%m-%d')

llm = ChatOpenAI(model='gpt-4.1-nano')

search_tool = TavilySearch(
    max_results=5,
    topic='general'
)

loader = PyMuPDFLoader('./data/흥국생명 실손의료비보험.pdf')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embedding=embedding)
retriever = vectorstore.as_retriever()

rag_tool = create_retriever_tool(
    retriever,
    name='pdf_search',
    description='PDF 문서에서 질문과 관련된 내용을 검색합니다.'  # Agent가 언제 이 tool을 쓸지 알게됨
)

text = f"""
너는 웹 검색이 가능하고, 흥국생명 상품 중 실손의료비보험 갱신형 상품 정보를 담은 pdf를 검색할 수 있는 어시스턴트야.

- 사용자가 문서 기반 질문(예: "이 PDF에서", "파일 내용", 특정 조항/보장/면책/예시)을 하면 반드시 pdf_search 도구를 사용해.
- 최신성/팩트체크가 필요하다고 판단되면 웹검색 도구를 사용해. 다만 문서와 충돌 시 문서를 우선하고, 차이가 있으면 둘 다 인용해.
- 답변에는 항상 핵심 결론 → 근거(인용) → 주의사항(있다면) 순서를 지켜.
- 인용은 [파일명 p.페이지] 형태로 표시하고, 여러 조각이면 ; 로 구분해.
- 숫자·비율·한도·면책 등 민감한 값은 반드시 인용 출처를 붙여.

오늘은 {today} 야.
"""


prompt = ChatPromptTemplate.from_messages([
    ('system', text),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad')  # 도구(검색) 호출때 필요함
])

memory = ConversationBufferMemory(
    return_messages=True,
    memory_key='chat_history'
)

agent = create_openai_tools_agent(
    llm=llm,
    tools=[search_tool, rag_tool],
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    memory=memory, 
    tools=[search_tool, rag_tool],
    verbose=True)


agent_executor.invoke({'input': '도수치료를 할 경우 본인부담금 비율이 어떻게 돼?'})

KeyboardInterrupt: 

In [ ]:
from operator import itemgetter
from pprint import pprint
from pydantic import BaseModel, Field


from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI


# Agent + RAG
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate as AgentPromptTemplate, MessagesPlaceholder as AgentMessagesPlaceholder
from langchain.memory import ConversationBufferMemory as AgentConversationBufferMemory
from langchain_tavily import TavilySearch
from datetime import datetime


# RAG 관련
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool

# LLM
llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)

# Output parser 정의
class Report(BaseModel):
    title: str = Field(..., description='답변의 제목')
    summary: str = Field(..., description='답변 요약본')
    content_kr: str = Field(..., description='한국어로 작성된 보고서의 내용(1000자 이내)')

parser = PydanticOutputParser(pydantic_object=Report)

# Prompt에 format instructions 추가
prompt = ChatPromptTemplate(
    [
        ('system',
         "너는 흥국생명 실손의료비보험 상품 정보를 담은 pdf를 검색해서 정보를 제공해주는 챗봇이야."
         "반드시 지정된 형식에 맞춰 JSON으로 답변해. 아래 PDF 컨텍스트를 우선 활용하고, 수치/비율/한도는 근거를 함께 제시해.\n\n"
         "FORMAT INSTRUCTION: {format_instructions}\n\n"
         "[PDF CONTEXT]\n{context}"),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{input}'),
    ]
).partial(format_instructions=parser.get_format_instructions())

# Memory
memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) |
    itemgetter('chat_history')
)

# 체인 구성
chain = runnable | prompt | llm | parser

# pdf 불러오기
loader = PyMuPDFLoader('./data/흥국생명 실손의료비보험.pdf')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embedding=embedding)
retriever = vectorstore.as_retriever()

rag_tool = create_retriever_tool(
    retriever,
    name='pdf_search',
    description='PDF 문서에서 질문과 관련된 내용을 검색합니다.'  # Agent가 언제 이 tool을 쓸지 알게됨
)

def build_pdf_context(query: str, k: int = 4) -> str:
    try:
        docs = retriever.get_relevant_documents(query)[:k]
    except Exception:
        return ""
    parts = []
    for d in docs:
        src = d.metadata.get('source', '')
        page = d.metadata.get('page') or d.metadata.get('page_number')
        # PyMuPDF는 보통 0-based라면 +1 보정, 아니면 그대로
        try:
            pnum = int(page) + 1 if isinstance(page, int) and page >= 0 else page
        except Exception:
            pnum = page
        fname = src.split('/')[-1] if src else 'PDF'
        cite = f"[{fname} p.{pnum}]" if pnum else f"[{fname}]"
        parts.append(f"{d.page_content.strip()}\n- 출처: {cite}")
    return "\n\n---\n\n".join(parts)

# 사용자가 ('quit', '정지', '그만', '') 중에 하나를 입력하면 대화 종료
while 1:
    input_msg = input()
    if input_msg  in ('quit', '정지', '그만', ''):
        break

    print('인간: ', input_msg)
    output_msg = chain.invoke({'input': input_msg})
    pprint(output_msg.model_dump())
    memory.save_context(
        {'human': input_msg},
        {'ai': output_msg.model_dump_json()}
    )

인간:  안녕
{'content_kr': '흥국생명 실손의료비보험은 의료비 실손 보장을 제공하는 보험 상품으로, 보험금 지급 한도는 병원비의 80%까지 '
               '보장하며, 연간 최대 지급 한도는 1억 원입니다. 입원과 통원 치료 모두 보장 대상이며, 본인 부담금은 병원비의 '
               '20%입니다. 또한, 특정 질병에 대한 특약도 선택 가능하며, 보험료는 연령과 건강 상태에 따라 차등 '
               '적용됩니다.',
 'summary': '흥국생명 실손의료비보험은 의료비 부담을 경감하기 위한 보험 상품으로, 다양한 보장 내용과 한도를 제공하고 있습니다.',
 'title': '흥국생명 실손의료비보험 안내'}
인간:  안녕
{'content_kr': '흥국생명 실손의료비보험은 의료비 실손 보장을 제공하는 보험 상품으로, 보험금 지급 한도는 병원비의 80%까지 '
               '보장하며, 연간 최대 지급 한도는 1억 원입니다. 입원과 통원 치료 모두 보장 대상이며, 본인 부담금은 병원비의 '
               '20%입니다. 또한, 특정 질병에 대한 특약도 선택 가능하며, 보험료는 연령과 건강 상태에 따라 차등 '
               '적용됩니다.',
 'summary': '흥국생명 실손의료비보험은 의료비 부담을 경감하기 위한 보험 상품으로, 다양한 보장 내용과 한도를 제공하고 있습니다.',
 'title': '흥국생명 실손의료비보험 안내'}


In [ ]:
# 그냥 챗봇

from operator import itemgetter
from pprint import pprint
from pydantic import BaseModel, Field

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

# Agent + RAG
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate as AgentPromptTemplate, MessagesPlaceholder as AgentMessagesPlaceholder
from langchain.memory import ConversationBufferMemory as AgentConversationBufferMemory
from langchain_tavily import TavilySearch
from datetime import datetime

# RAG 관련
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool


# --------------------------- 공통 LLM ---------------------------
llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)


# --------------------------- 리포트(JSON) 체인 ---------------------------
class Report(BaseModel):
    title: str = Field(..., description='답변의 제목')
    summary: str = Field(..., description='답변 요약본')
    content_kr: str = Field(..., description='한국어로 작성된 보고서의 내용(1000자 이내)')

parser = PydanticOutputParser(pydantic_object=Report)

# 프롬프트: FORMAT INSTRUCTION + (context 기본값은 빈 문자열로 partial)
prompt = ChatPromptTemplate(
    [
        ('system',
         "너는 웹 검색이 가능하고, 흥국생명 실손의료비보험 상품 정보를 담은 pdf를 검색해서 정보를 제공해주는 챗봇이야."
         " 반드시 지정된 형식에 맞춰 JSON으로 답변해.\n\n"
         "FORMAT INSTRUCTION: {format_instructions}\n\n"
         "[PDF CONTEXT]\n{context}"),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{input}'),
    ]
).partial(format_instructions=parser.get_format_instructions(), context="")

# Memory (리포트 체인용)
memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) | itemgetter('chat_history')
)

# 리포트 체인
chain = runnable | prompt | llm | parser


# --------------------------- PDF 인덱스 & 도구 ---------------------------
# Tavily는 키 없을 때 예외가 날 수 있으므로 가드
try:
    search_tool = TavilySearch(max_results=5, topic='general')
except Exception:
    search_tool = None

# PDF → 분할 → 임베딩 → FAISS → Retriever
retriever = None
rag_tool = None
try:
    loader = PyMuPDFLoader('./data/흥국생명 실손의료비보험.pdf')
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    split_docs = splitter.split_documents(docs)
    embedding = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(split_docs, embedding=embedding)
    retriever = vectorstore.as_retriever()

    rag_tool = create_retriever_tool(
        retriever,
        name='pdf_search',
        description='PDF 문서에서 질문과 관련된 내용을 검색합니다.'  # Agent가 언제 이 tool을 쓸지 알게됨
    )
except Exception as e:
    print(f"[알림] PDF 인덱싱을 건너뜁니다: {e}")


# --------------------------- 에이전트(대화 모드) ---------------------------
_today = datetime.today().strftime('%Y-%m-%d')
agent_system = (
    f"너는 PDF(RAG)와 (가능하면) 웹검색 도구를 사용하는 어시스턴트야. "
    f"문서 기반 질문이면 반드시 pdf_search를 사용하고, 수치/비율/한도는 근거를 덧붙여라. 오늘은 {_today}."
)

agent_prompt = AgentPromptTemplate.from_messages([
    ('system', agent_system),
    AgentMessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    AgentMessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent_memory = AgentConversationBufferMemory(return_messages=True, memory_key='chat_history')
tools = [t for t in [rag_tool, search_tool] if t is not None]

agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt,
)
agent_executor = AgentExecutor(
    agent=agent,
    memory=agent_memory,
    tools=tools,
    verbose=True
)


# --------------------------- 런 루프 ---------------------------
# 사용자가 ('quit', '정지', '그만', '') 중에 하나를 입력하면 대화 종료
# '/report <질문>' → 리포트(JSON) 체인 사용
# 그 외 입력 → 에이전트(RAG/웹검색) 대화
while 1:
    input_msg = input()
    if input_msg in ('quit', '정지', '그만', ''):
        break

    # 리포트(JSON) 모드
    if input_msg.startswith('/report '):
        q = input_msg.replace('/report', '', 1).strip()
        print('인간: ', q)
        output_msg = chain.invoke({'input': q})
        pprint(output_msg.model_dump())
        memory.save_context({'human': q}, {'ai': output_msg.model_dump_json()})
        continue

    # 대화(RAG/웹검색) 모드
    print('인간: ', input_msg)
    try:
        result = agent_executor.invoke({'input': input_msg})
        print(result.get('output', result))
    except Exception as e:
        print(f'[에이전트 오류] {e}')


인간:  안녕


> Entering new AgentExecutor chain...
안녕하세요! 무엇을 도와드릴까요?

> Finished chain.
안녕하세요! 무엇을 도와드릴까요?
인간:  안녕


> Entering new AgentExecutor chain...
안녕하세요! 다시 인사해 주셔서 감사합니다. 오늘은 어떤 도움을 드릴까요?

> Finished chain.
안녕하세요! 다시 인사해 주셔서 감사합니다. 오늘은 어떤 도움을 드릴까요?
인간:  도수치료를 받고 싶은데 내가 얼마나 부담해야해?


> Entering new AgentExecutor chain...
도수치료 비용은 지역, 병원 또는 클리닉, 치료의 종류와 강도에 따라 다를 수 있습니다. 일반적으로 도수치료의 비용은 1회당 3만 원에서 10만 원 정도로 형성되어 있으며, 보험 적용 여부에 따라 본인 부담액이 달라질 수 있습니다.

한국에서는 건강보험이 일부 적용될 경우, 본인 부담금은 전체 비용의 30% 정도가 될 수 있습니다. 예를 들어, 1회 치료 비용이 5만 원이라면, 보험 적용 시 본인 부담금은 약 1만 5천 원 정도가 될 수 있습니다.

정확한 비용 정보를 위해서는 가까운 병원이나 클리닉에 문의하는 것이 가장 좋습니다. 혹시 더 구체적인 정보를 원하시면, 지역이나 병원 이름을 알려주시면 검색해서 도와드릴 수 있습니다.

> Finished chain.
도수치료 비용은 지역, 병원 또는 클리닉, 치료의 종류와 강도에 따라 다를 수 있습니다. 일반적으로 도수치료의 비용은 1회당 3만 원에서 10만 원 정도로 형성되어 있으며, 보험 적용 여부에 따라 본인 부담액이 달라질 수 있습니다.

한국에서는 건강보험이 일부 적용될 경우, 본인 부담금은 전체 비용의 30% 정도가 될 수 있습니다. 예를 들어, 1회 치료 비용이 5만 원이라면, 보험 적용 시 본인 부담금은 약 1만 5천 원 정도가 될 수 있습니다.

정확한 비용 정보를 위해서는 가까운 병원이나 클리닉에 문의하는 것이 가장 좋

In [ ]:
# 실시간 흥국생명 실손보험 약관 전용 챗봇

from operator import itemgetter
from pprint import pprint
from pydantic import BaseModel, Field

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

# Agent + RAG
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate as AgentPromptTemplate, MessagesPlaceholder as AgentMessagesPlaceholder
from langchain.memory import ConversationBufferMemory as AgentConversationBufferMemory
from langchain_tavily import TavilySearch
from datetime import datetime

# RAG 관련
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool
from dotenv import load_dotenv
load_dotenv()

# LLM
llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)


# JSON 체인
class Report(BaseModel):
    title: str = Field(..., description='답변의 제목')
    summary: str = Field(..., description='답변 요약본')
    content_kr: str = Field(..., description='한국어로 작성된 보고서의 내용(1000자 이내)')

parser = PydanticOutputParser(pydantic_object=Report)

prompt = ChatPromptTemplate(
    [
        ('system',
         "너는 웹 검색이 가능하고, 흥국생명 실손의료비보험 약관 PDF를 근거로 정보를 제공하는 리포트 생성기야. "
         "반드시 지정된 형식에 맞춰 JSON으로 답변해.\n\n"
         "FORMAT INSTRUCTION: {format_instructions}\n\n"
         "[PDF CONTEXT]\n{context}"),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{input}'),
    ]
).partial(format_instructions=parser.get_format_instructions(), context="")

# Memory
memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')

runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) | itemgetter('chat_history')
)

# 체인
chain = runnable | prompt | llm | parser


# --------------------------- PDF 인덱스 & 도구 ---------------------------
try:
    search_tool = TavilySearch(max_results=5, topic='general')
except Exception:
    search_tool = None

# PDF → 분할 → 임베딩 → FAISS → Retriever
retriever = None
rag_tool = None
try:
    loader = PyMuPDFLoader('./data/흥국생명 실손의료비보험.pdf')
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    split_docs = splitter.split_documents(docs)
    embedding = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(split_docs, embedding=embedding)
    # 추출 다양성/중복 방지 효과를 위해 k 소폭 상향해도 됨(최소 변경 원칙으로 기본 유지)
    retriever = vectorstore.as_retriever()

    # 약관 전용 툴 설명 강화
    rag_tool = create_retriever_tool(
        retriever,
        name='pdf_search',
        description=(
            "흥국생명 실손의료비보험 약관 PDF에서만 관련 조항/보장/면책/한도/본인부담 정보를 검색합니다. "
            "약관 근거가 필요한 모든 질문에서 이 도구를 먼저 호출하세요. 반환된 원문은 참고용이며, 최종 답변에 원문을 그대로 출력하지 말고 정리하세요. "
            "가능하면 페이지 번호와 핵심 문구를 근거로 제시하세요."
        )
    )
except Exception as e:
    print(f"[알림] PDF 인덱싱을 건너뜁니다: {e}")


# --------------------------- 에이전트(대화 모드) ---------------------------
_today = datetime.today().strftime('%Y-%m-%d')

# ✨ 핵심: 깔끔한 출력 템플릿 + 원문 그대로 출력 금지
agent_system = f"""
너는 '흥국생명 실손의료비보험 약관 챗봇 에이전트'고 답변의 1차 근거는 제공된 약관 PDF뿐이야.

[역할/원칙]
- 매 질문마다 pdf_search를 먼저 호출해 약관 근거를 확보
- 도구가 반환한 원문을 그대로 출력하지 말고, 한국어로 매끄럽게 요약·정리
- 수치/비율/금액/기간 등은 근거를 함께 제시 (예: [흥국생명_실손의료비보험.pdf p.12])
- 약관 범위 밖(타사 비교/보험료 견적/최신 공시 등)은 범위 밖임을 알리고, 원할 때만 웹검색을 제안
- 디버그/도구 호출 로그/원문 나열/코드블록은 출력 금지

[응답 형식]
1) 핵심요약: 한 줄
2) 자기부담/한도: 관련 수치·조건
3) 상세설명: 적용 범위·예외·필수 요건
4) 면책/비보장: 해당 시 명시
5) 예시 계산: 가능한 경우만
6) 근거: [파일명 p.xx]; 여러 개면 세미콜론 구분

오늘은 {_today}.
"""

agent_prompt = AgentPromptTemplate.from_messages([
    ('system', agent_system),
    AgentMessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    AgentMessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent_memory = AgentConversationBufferMemory(return_messages=True, memory_key='chat_history')
tools = [t for t in [rag_tool, search_tool] if t is not None]

agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt,
)
# 🔇 깔끔한 대화 출력
agent_executor = AgentExecutor(
    agent=agent,
    memory=agent_memory,
    tools=tools,
    verbose=False
)


# 챗봇시작
while 1:
    input_msg = input()
    if input_msg in ('quit', '정지', '그만', ''):
        break

    # 리포트(JSON) 모드
    if input_msg.startswith('/report '):
        q = input_msg.replace('/report', '', 1).strip()
        print('인간: ', q)
        output_msg = chain.invoke({'input': q})
        pprint(output_msg.model_dump())
        memory.save_context({'human': q}, {'ai': output_msg.model_dump_json()})
        continue

    # 대화(약관 전용) 모드
    print('인간: ', input_msg)
    try:
        result = agent_executor.invoke({'input': input_msg})
        print(result.get('output', result))
    except Exception as e:
        print(f'[에이전트 오류] {e}')


C:\Users\samsung\AppData\Local\Temp\ipykernel_21168\2564008248.py:54: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True, memory_key='chat_history')


인간:  안녕
안녕하세요! 흥국생명 실손의료비보험 약관에 대해 궁금하신 점이 있으시면 도와드리겠습니다. 약관은 보험 계약의 내용과 조건을 규정하는 중요한 문서로, 이해를 돕기 위해 다양한 안내 자료와 요약서, 영상, QR코드 등을 제공하고 있습니다. 구체적인 내용이나 궁금한 점이 있으시면 말씀해 주세요.
인간:  보험 해지시 약관 정리
1) 핵심요약: 보험 해지와 관련된 약관은 계약자의 임의 해지, 법적 위반, 중대한 사유에 따른 해지 조건과 해약환급금 지급 규정을 포함하고 있습니다.
2) 자기부담/한도: 계약자는 언제든지 특약을 해지할 수 있으며, 해지 시 해약환급금을 받을 수 있습니다(약관에 따라 다름). 회사는 해지 요청 후 10일 이내에 수락 여부를 통지하며, 정당한 사유 없이 해지 요청을 거부할 수 없습니다.
3) 상세설명: 계약자는 언제든지 특약을 해지할 수 있으며, 해지 시 해약환급금을 지급받을 수 있습니다. 출생예정일 이후 또는 태아인 경우 출생 전 해지 시, 의사소견서 등 증빙서류를 제출해야 합니다. 계약자가 고의 또는 중과실로 사실과 다르게 알린 경우, 또는 법률 위반 시 계약 해지가 가능하며, 회사는 해지 후 일정 조건에 따라 해약환급금을 지급합니다. 회사는 해지 요청 후 10일 이내에 수락 여부를 통지하며, 정당한 사유 없이 해지 요청을 거부할 수 없습니다.
4) 면책/비보장: 계약자가 고의 또는 중과실로 사실과 다르게 알린 경우, 또는 법률 위반으로 인한 계약 해지는 면책 사유입니다.
5) 예시 계산: 해지 시점에 따라 해약환급금이 다를 수 있으며, 구체적 금액은 약관에 따라 결정됩니다(약관 상세 내용 참조).
6) 근거: [흥국생명_실손의료비보험.pdf p.25-28]
인간:  실손보험 청구시 필요 서류
1) 핵심요약: 실손보험 청구 시에는 보험금 청구서, 진단서, 의료비 영수증, 통원 또는 입원확인서, 신분증, 가족관계서류 등 다양한 서류를 제출해야 합니다.
2) 자기부담/한도: 별도 자기부담금이나 한도는 약관에 명시되어 있지 않으며